# Dataset Oversampling and Copy-Paste Augmentation

This notebook implements a "Copy-Paste" data augmentation strategy designed to balance class distributions in instance segmentation datasets. It specifically targets minority classes by extracting their instances and pasting them onto images containing majority classes (backgrounds), while applying constrained transformations to preserve realism.

### Dataset Statistics
- Silt: 33.02% (Majority)
- Shalestone: 19.32%
- Coal: 18.56%
- Quartz: 10.72% (Minority)
- Limestone: 10.22% (Minority)
- Sandstone: 8.17% (Minority)

### Augmentation Constraints & Guidelines
To prevent generating unrealistic "overkill" images, the following rules are applied:
1. **Random Scale Jittering**: 0.8x to 1.25x scaling.
2. **Horizontal Flip**: 50% probability.
3. **Brightness Adjustment**: \pm 20% alteration.
4. **No Semantic Rotation/Hue Changes**: Preserves natural directional textures and critical color distributions.
5. **Constraints**: Maximum of 15 pasted objects per image, occupying at most 30-40% of the total area.
6. **Edge Smoothing**: Applies Gaussian blur to object masks prior to pasting to eliminate sharp, artificial edges.
7. **Occlusion Handling**: Allows objects to overlap, accurately updating segmentation masks to reflect visible areas.

In [1]:
import os
import cv2
import json
import random
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

## 1. Configuration

In [2]:
INPUT_DATASET_DIR = "/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO/train"
OUTPUT_DATASET_DIR = "/home/praktikan/projects/DwiAnggara/Datasets/Batch3and4_YOLO_Oversampled_200/train"
DATASET_FORMAT = "YOLO" # Support "YOLO" or "COCO"

CLASS_MAP = {
    0: "Silt",
    1: "Sandstone",
    2: "Limestone",
    3: "Coal",
    4: "Shalestone",
    5: "Quartz",
}
MINORITY_CLASSES = [1, 2, 5]
MAJORITY_CLASSES = [0]

AUGMENTATION_CONFIG = {
    "scale_min": 0.8,
    "scale_max": 1.25,
    "flip_prob": 0.5,
    "brightness_jitter": 0.2,
    "max_paste_per_image": 25,
    "max_area_ratio": 0.45,
    "edge_blur_kernel": (3, 3)
}

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

ensure_dir(os.path.join(OUTPUT_DATASET_DIR, "images"))
ensure_dir(os.path.join(OUTPUT_DATASET_DIR, "labels"))


## 2. Utility Functions for Mask & Polygon Processing

In [3]:
def yolo_to_polygon(yolo_coords, width, height):
    """
    Converts normalized YOLO segmentation coordinates to pixel polygons.
    """
    pts = [float(p) for p in yolo_coords]
    pts_xy = np.array([(pts[i] * width, pts[i+1] * height) for i in range(0, len(pts)-1, 2)], dtype=np.int32)
    return pts_xy

def polygon_to_yolo(contours, width, height):
    """
    Converts pixel contours back to normalized YOLO strings.
    """
    yolo_polygons = []
    for contour in contours:
        if len(contour) >= 3:
            contour = contour.squeeze()
            if contour.ndim == 1:
                continue
            norm_pts = []
            for point in contour:
                x_norm = max(0.0, min(1.0, point[0] / width))
                y_norm = max(0.0, min(1.0, point[1] / height))
                norm_pts.extend([f"{x_norm:.6f}", f"{y_norm:.6f}"])
            yolo_polygons.append(" ".join(norm_pts))
    return yolo_polygons

def extract_object_from_image(image, polygon):
    """
    Extracts an object from an image based on its polygon boundary,
    returning the RGBA cutout and its local mask.
    """
    x, y, w, h = cv2.boundingRect(polygon)
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [polygon], 255)

    obj_img = image[y:y+h, x:x+w]
    obj_mask = mask[y:y+h, x:x+w]
    
    # Append mask as alpha channel
    b, g, r = cv2.split(obj_img)
    rgba_obj = cv2.merge([b, g, r, obj_mask])
    return rgba_obj, obj_mask

## 3. Augmentation Core

In [4]:
def apply_object_augmentation(obj_rgba):
    """
    Applies scaling, flipping, and brightness adjustments to an extracted object.
    """
    # 1. Random Scale
    scale = random.uniform(AUGMENTATION_CONFIG["scale_min"], AUGMENTATION_CONFIG["scale_max"])
    h, w = obj_rgba.shape[:2]
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    obj_scaled = cv2.resize(obj_rgba, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # 2. Horizontal Flip
    if random.random() < AUGMENTATION_CONFIG["flip_prob"]:
        obj_scaled = cv2.flip(obj_scaled, 1)

    # 3. Brightness Adjustment
    b, g, r, alpha = cv2.split(obj_scaled)
    rgb_img = cv2.merge([b, g, r])
    
    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2HSV)
    hsv = np.array(hsv, dtype=np.float64)
    brightness_factor = 1.0 + random.uniform(-AUGMENTATION_CONFIG["brightness_jitter"], AUGMENTATION_CONFIG["brightness_jitter"])
    hsv[:, :, 2] = hsv[:, :, 2] * brightness_factor
    hsv[:, :, 2][hsv[:, :, 2] > 255] = 255
    hsv = np.array(hsv, dtype=np.uint8)
    rgb_adjusted = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    
    # 4. Edge Feathering (Gaussian Blur on Alpha channel)
    alpha_blurred = cv2.GaussianBlur(alpha, AUGMENTATION_CONFIG["edge_blur_kernel"], 0)
    
    b, g, r = cv2.split(rgb_adjusted)
    final_obj = cv2.merge([b, g, r, alpha_blurred])
    
    return final_obj

## 4. Copy-Paste Pipeline & Occlusion Handling

In [5]:
def extract_minority_objects(dataset_dir, fmt="YOLO"):
    """
    Scans the dataset to build a bank of minority class objects.
    """
    labels_dir = os.path.join(dataset_dir, "labels")
    images_dir = os.path.join(dataset_dir, "images")
    object_bank = []
    majority_backgrounds = []

    if fmt == "YOLO":
        label_files = list(Path(labels_dir).glob("*.txt"))
        for lbl_path in tqdm(label_files):
            img_name = lbl_path.stem + ".png"
            img_path = os.path.join(images_dir, img_name)
            if not os.path.exists(img_path):
                img_name = lbl_path.stem + ".jpg"
                img_path = os.path.join(images_dir, img_name)
                if not os.path.exists(img_path): continue

            image = cv2.imread(str(img_path))
            if image is None: continue
            h, w = image.shape[:2]
            has_majority = False
            
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    if class_id in MAJORITY_CLASSES: has_majority = True
                    if class_id in MINORITY_CLASSES:
                        polygon = yolo_to_polygon(parts[1:], w, h)
                        obj_rgba, obj_mask = extract_object_from_image(image, polygon)
                        if cv2.countNonZero(obj_mask) > 100:
                            object_bank.append({"class_id": class_id, "rgba": obj_rgba, "source": img_name})
            if has_majority: majority_backgrounds.append(img_path)
            
    elif fmt == "COCO":
        import json
        coco_json_path = os.path.join(labels_dir, "instances_default.json")
        if os.path.exists(coco_json_path):
            with open(coco_json_path, 'r') as f:
                coco_data = json.load(f)
            
            img_dict = {img['id']: img['file_name'] for img in coco_data['images']}
            for img_id, img_name in tqdm(img_dict.items()):
                img_path = os.path.join(images_dir, img_name)
                if not os.path.exists(img_path): continue
                
                image = cv2.imread(str(img_path))
                if image is None: continue
                has_majority = False
                
                anns = [a for a in coco_data['annotations'] if a['image_id'] == img_id]
                for ann in anns:
                    class_id = ann['category_id']
                    if class_id in MAJORITY_CLASSES: has_majority = True
                    if class_id in MINORITY_CLASSES:
                        seg = ann['segmentation'][0]
                        polygon = np.array([(seg[i], seg[i+1]) for i in range(0, len(seg)-1, 2)], dtype=np.int32)
                        obj_rgba, obj_mask = extract_object_from_image(image, polygon)
                        if cv2.countNonZero(obj_mask) > 100:
                            object_bank.append({"class_id": class_id, "rgba": obj_rgba, "source": img_name})
                if has_majority: majority_backgrounds.append(img_path)

    return object_bank, majority_backgrounds

def execute_copy_paste_pipeline(dataset_dir, output_dir, object_bank, backgrounds, num_synthetic_images=500, fmt="YOLO"):
    for i in tqdm(range(num_synthetic_images), desc="Generating Synthetic Images"):
        bg_path = random.choice(backgrounds)
        bg_img = cv2.imread(str(bg_path))
        bg_h, bg_w = bg_img.shape[:2]
        
        instance_mask_matrix = np.zeros((bg_h, bg_w), dtype=np.int32)
        class_map = {}
        current_instance_id = 1
        
        # Existing Annotations
        if fmt == "YOLO":
            lbl_path = os.path.join(dataset_dir, "labels", Path(bg_path).stem + ".txt")
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    for line in f.readlines():
                        parts = line.strip().split()
                        cls_id, poly = int(parts[0]), yolo_to_polygon(parts[1:], bg_w, bg_h)
                        cv2.fillPoly(instance_mask_matrix, [poly], current_instance_id)
                        class_map[current_instance_id] = cls_id
                        current_instance_id += 1
                        
        num_paste = random.randint(3, AUGMENTATION_CONFIG["max_paste_per_image"])
        accumulated_area = 0
        max_area = bg_w * bg_h * AUGMENTATION_CONFIG["max_area_ratio"]
        used_indices = set()
        
        for _ in range(num_paste):
            obj_idx = random.randint(0, len(object_bank) - 1)
            if obj_idx in used_indices: continue
            used_indices.add(obj_idx)
            
            obj_data = object_bank[obj_idx]
            aug_obj = apply_object_augmentation(obj_data["rgba"])
            obj_h, obj_w = aug_obj.shape[:2]
            
            if obj_h >= bg_h or obj_w >= bg_w: continue
            obj_area = cv2.countNonZero(aug_obj[:, :, 3])
            if accumulated_area + obj_area > max_area: break
                
            max_y, max_x = bg_h - obj_h, bg_w - obj_w
            paste_y, paste_x = random.randint(0, max_y), random.randint(0, max_x)
            
            bg_roi = bg_img[paste_y:paste_y+obj_h, paste_x:paste_x+obj_w]
            alpha_mask = aug_obj[:, :, 3] / 255.0
            for c in range(3):
                bg_roi[:, :, c] = (1. - alpha_mask) * bg_roi[:, :, c] + alpha_mask * aug_obj[:, :, c]
            bg_img[paste_y:paste_y+obj_h, paste_x:paste_x+obj_w] = bg_roi
            
            binary_mask = (aug_obj[:, :, 3] > 128).astype(np.uint8)
            roi_instance = instance_mask_matrix[paste_y:paste_y+obj_h, paste_x:paste_x+obj_w]
            roi_instance[binary_mask == 1] = current_instance_id
            instance_mask_matrix[paste_y:paste_y+obj_h, paste_x:paste_x+obj_w] = roi_instance
            class_map[current_instance_id] = obj_data["class_id"]
            
            current_instance_id += 1
            accumulated_area += obj_area

        output_image_path = os.path.join(output_dir, "images", f"synth_{i}.jpg")
        cv2.imwrite(output_image_path, bg_img)
        
        if fmt == "YOLO":
            output_label_path = os.path.join(output_dir, "labels", f"synth_{i}.txt")
            with open(output_label_path, 'w') as f:
                for inst_id, cls_id in class_map.items():
                    inst_mask = (instance_mask_matrix == inst_id).astype(np.uint8) * 255
                    if cv2.countNonZero(inst_mask) < 50: continue
                    contours, _ = cv2.findContours(inst_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    for y_str in polygon_to_yolo(contours, bg_w, bg_h):
                        f.write(f"{cls_id} {y_str}\n")


## 5. Execution

In [6]:
if __name__ == "__main__":
    print("Initializing Pipeline...")
    obj_bank, bg_candidates = extract_minority_objects(INPUT_DATASET_DIR, fmt=DATASET_FORMAT)
    
    if obj_bank and bg_candidates:
        execute_copy_paste_pipeline(INPUT_DATASET_DIR, OUTPUT_DATASET_DIR, obj_bank, bg_candidates, num_synthetic_images=200, fmt=DATASET_FORMAT)
    else:
        print("Data extraction failed. Please check paths and category IDs.")


Initializing Pipeline...


Generating Synthetic Images: 100%|██████████| 200/200 [02:18<00:00,  1.44it/s]
